<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/05_joins.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 · Joining tables (JOIN)

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Using SQLs for omics: basics and hands-on* - Instructor: Anderson Santos

**Estimated time: ~45 min**

## Learning objectives
- combine tables with `INNER JOIN ... ON`;
- join three tables (`abundance` ↔ `samples` ↔ `taxa`);
- keep unmatched rows with `LEFT JOIN` (and spot NULLs).

Real questions need columns from several tables at once. A JOIN combines rows from two tables that share a key. This is the core of relational SQL.

---
| # | Lesson |
|---|---|
| 00 | Database foundations (concepts) |
| 01 | Meet the database & first SELECT |
| 02 | Filtering rows (WHERE) |
| 03 | Sorting, limiting & computed columns |
| 04 | Aggregation (GROUP BY) |
| 05 | Joining tables (JOIN) |
| 06 | Subqueries & CTEs |
| 07 | Complete & complex queries |
| 08 | Hands-on capstone |
| 09 | Bonus: creating & changing data |

## Setup — run this cell first

This connects the notebook to the example database. It works both on your own computer and on Google Colab; just run it (Shift+Enter). You do not need to understand it.

In [1]:
# Run me first. Works locally AND on Google Colab.
%pip install jupysql --quiet
import os
REPO = "andersonavilasantos/ufz-summerschool-sql"   # course repository
if not os.path.exists("../data/omics.db"):
    # Not found locally -> on Colab: download the course and enter it.
    os.system(f"git clone -q https://github.com/{REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")
%load_ext sql
%sql sqlite:///../data/omics.db
print("Connected to omics.db — you are ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.6/397.6 kB 20.0 MB/s eta 0:00:00


Connecting to 'sqlite:///../data/omics.db'

Connected to omics.db — you are ready.


## 1. Join abundance to taxa
The `abundance` table only stores IDs. Join `taxa` to get the phylum, then total the reads per phylum.

In [3]:
%%sql
-- did it myself
SELECT *
from abundance



Running query in 'sqlite:///../data/omics.db'

sample_id,taxon_id,count
S001,T001,9
S002,T001,47
S003,T001,10
S004,T001,14
S005,T001,16
S006,T001,49
S007,T001,3
S008,T001,11
S009,T001,25
S010,T001,38


In [4]:
%%sql
-- did it myself
SELECT *
from samples



Running query in 'sqlite:///../data/omics.db'

sample_id,site,environment,treatment,ph,temperature_c,depth_cm,collection_date
S001,LEI_SOIL,Soil,Control,5.88,14.5,9.3,2025-04-11
S002,LEI_SOIL,Soil,Control,6.46,13.4,18.1,2025-08-05
S003,LEI_SOIL,Soil,Control,6.09,13.5,19.3,2025-08-25
S004,LEI_SOIL,Soil,Amended,6.17,17.8,None,2025-07-23
S005,LEI_SOIL,Soil,Amended,6.04,19.1,5.6,2025-04-07
S006,LEI_SOIL,Soil,Amended,6.01,19.3,13.3,2025-08-01
S007,UPP_SOIL,Soil,Control,6.46,17.6,3.9,2025-06-17
S008,UPP_SOIL,Soil,Control,5.71,14.3,4.2,2025-09-24
S009,UPP_SOIL,Soil,Control,None,16.7,6.9,2025-06-26
S010,UPP_SOIL,Soil,Amended,5.85,19.4,18.0,2025-04-09


In [8]:
%%sql
SELECT t.phylum, SUM(a.count) AS reads
FROM abundance a
JOIN taxa t ON a.taxon_id = t.taxon_id
GROUP BY t.phylum
ORDER BY reads DESC;

Running query in 'sqlite:///../data/omics.db'

phylum,reads
Pseudomonadota,7540
Actinomycetota,6298
Bacillota,5805
Acidobacteriota,4102
Bacteroidota,3879
Euryarchaeota,3231
Nitrospirota,2617
Cyanobacteriota,2329


The same join, a different question: total reads per domain (Bacteria vs. Archaea).

In [9]:
%%sql
-- join to taxa, then summarise by the broader 'domain' rank
SELECT t.domain, SUM(a.count) AS reads
FROM abundance a
JOIN taxa t ON a.taxon_id = t.taxon_id
GROUP BY t.domain
ORDER BY reads DESC;

Running query in 'sqlite:///../data/omics.db'

domain,reads
Bacteria,32570
Archaea,3231


## 2. Join abundance to samples
Which environment has the most reads?

In [6]:
%%sql
SELECT s.environment, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
GROUP BY s.environment
ORDER BY reads DESC;

Running query in 'sqlite:///../data/omics.db'

environment,reads
Soil,15268
Sediment,10616
Freshwater,9917


Join to `samples` to summarise by a different sample attribute, here the treatment.

In [7]:
%%sql
-- reads per treatment (Control vs. Amended) via the same join
SELECT s.treatment, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
GROUP BY s.treatment
ORDER BY reads DESC;

Running query in 'sqlite:///../data/omics.db'

treatment,reads
Amended,22495
Control,13306


## 3. Three-table join
Phylum composition per environment: this recovers the ecology (Cyanobacteria in freshwater, methanogens in sediment…).

In [10]:
%%sql
SELECT s.environment, t.phylum, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN taxa t    ON a.taxon_id  = t.taxon_id
GROUP BY s.environment, t.phylum
ORDER BY s.environment, reads DESC;

Running query in 'sqlite:///../data/omics.db'

environment,phylum,reads
Freshwater,Pseudomonadota,2919
Freshwater,Cyanobacteriota,1963
Freshwater,Bacteroidota,1534
Freshwater,Bacillota,1110
Freshwater,Actinomycetota,1013
Freshwater,Nitrospirota,615
Freshwater,Acidobacteriota,418
Freshwater,Euryarchaeota,345
Sediment,Euryarchaeota,2457
Sediment,Pseudomonadota,1757


In [13]:
%%sql
SELECT * FROM abundance a

Running query in 'sqlite:///../data/omics.db'

sample_id,taxon_id,count
S001,T001,9
S002,T001,47
S003,T001,10
S004,T001,14
S005,T001,16
S006,T001,49
S007,T001,3
S008,T001,11
S009,T001,25
S010,T001,38


## 4. Bring in the sites table
Reads per country (four-table reach via keys).

In [14]:
%%sql
SELECT si.country, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN sites si  ON s.site = si.site
GROUP BY si.country
ORDER BY reads DESC;

Running query in 'sqlite:///../data/omics.db'

country,reads
Germany,17480
Sweden,7559
Denmark,5705
Norway,5057


## 5. LEFT JOIN — keep rows with no match
List every taxon and its count in sample `S001`; taxa absent from S001 show `NULL` (an INNER JOIN would drop them).

In [17]:
%%sql
SELECT t.genus, a.count
FROM taxa t
LEFT JOIN abundance a ON t.taxon_id = a.taxon_id AND a.sample_id = 'S001'
ORDER BY a.count DESC;

Running query in 'sqlite:///../data/omics.db'

genus,count
Mycobacterium,124
Streptomyces,97
Streptomyces,75
Acidobacterium,74
Clostridium,59
Chryseobacterium,45
Paenibacillus,42
Terriglobus,42
Arthrobacter,37
Mycobacterium,34


A LEFT JOIN plus `WHERE ... IS NULL` is the classic way to find the non-matches, here taxa never seen in S001.

In [ ]:
%%sql
-- taxa completely ABSENT from sample S001 (the unmatched left rows)
SELECT t.genus, t.phylum
FROM taxa t
LEFT JOIN abundance a ON t.taxon_id = a.taxon_id AND a.sample_id = 'S001'
WHERE a.taxon_id IS NULL
ORDER BY t.genus;

---
## Exercises

**Exercise.** Total reads per genus (join abundance to taxa); show the top 10.

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT t.genus, SUM(a.count) AS reads
FROM abundance a
JOIN taxa t ON a.taxon_id = t.taxon_id
GROUP BY t.genus
ORDER BY reads DESC
LIMIT 10;
```
</details>

**Exercise.** Reads per environment for the domain `Archaea` only (join all three, filter on domain).

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT s.environment, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN taxa t    ON a.taxon_id  = t.taxon_id
WHERE t.domain = 'Archaea'
GROUP BY s.environment
ORDER BY reads DESC;
```
</details>

**Exercise.** Which single phylum has the most reads in `Sediment` samples?

In [ ]:
%%sql
-- write your query here

<details>
<summary><b>Solution</b> (click to expand)</summary>

```sql
SELECT t.phylum, SUM(a.count) AS reads
FROM abundance a
JOIN samples s ON a.sample_id = s.sample_id
JOIN taxa t    ON a.taxon_id  = t.taxon_id
WHERE s.environment = 'Sediment'
GROUP BY t.phylum
ORDER BY reads DESC
LIMIT 1;
```
</details>

### Recap
- `JOIN b ON a.key = b.key` combines tables that share a key.
- Chain multiple `JOIN`s to reach three or four tables.
- `LEFT JOIN` keeps unmatched left-side rows (their right columns are NULL).
- Give tables short aliases (`a`, `s`, `t`) and prefix columns.
- Next: 06 · Subqueries & CTEs.